Language: **English (default)** | [Polski](lab.ipynb)

# Lab 12 -- anomaly detection with an autoencoder

Companion material to [`README.md`](README.md). Dataset: Fashion-MNIST,
normal class = *Sneaker* (7), anomalies = the remaining 9 classes.
**Not** the same task as the lecture (there: KMNIST,
reconstruction/generation) -- here: anomaly detection, with an emphasis
on when this method fails.

In [1]:
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt  # noqa: F401 - used in the TODO below
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.datasets as tvd
import torchvision.transforms as T

_REPO_ROOT = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "pyproject.toml").is_file()
    ),
    Path.cwd(),
)
DATA_CACHE_DIR = Path(
    os.environ.get("DEEP_LEARNING_DATA_CACHE", _REPO_ROOT / "data_cache")
).expanduser().resolve()

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Using device: {device}")

CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
NORMAL_CLASS = 7  # Sneaker

train_full = tvd.FashionMNIST(
    root=DATA_CACHE_DIR, train=True, download=True, transform=T.ToTensor()
)
test_full = tvd.FashionMNIST(
    root=DATA_CACHE_DIR, train=False, download=True, transform=T.ToTensor()
)

train_imgs = torch.stack([train_full[i][0] for i in range(len(train_full))])
train_labels = torch.tensor([train_full[i][1] for i in range(len(train_full))])
test_imgs = torch.stack([test_full[i][0] for i in range(len(test_full))])
test_labels = torch.tensor([test_full[i][1] for i in range(len(test_full))])

print(f"train: {train_imgs.shape}, test: {test_imgs.shape}")
print(f"normal class: {NORMAL_CLASS} ({CLASS_NAMES[NORMAL_CLASS]})")

D_IN = 28 * 28

Using device: cpu


train: torch.Size([60000, 1, 28, 28]), test: torch.Size([10000, 1, 28, 28])
normal class: 7 (Sneaker)


## 1. Data and split -- your task

Build **five** disjoint subsets (see `README.md`, section 1, for full
instructions):

1. `ae_train_X` -- only the normal class, from `train_imgs`/`train_labels`.
2. `val_normal_X` -- **other** normal-class examples, from `train_imgs`
   (disjoint from `ae_train_X`).
3. `val_anomaly_X` -- a sample of other classes, from `train_imgs`.
4. `test_normal_X` -- the normal class, from `test_imgs`/`test_labels` (a
   separate, official test set).
5. `test_anomaly_X` (+ `test_anomaly_labels` -- useful for per-class error
   analysis in section 6) -- a sample of other classes, from `test_imgs`.

**Verify programmatically** that `ae_train_X`/`val_normal_X`/`val_anomaly_X`
do not overlap by index (not just by comparing sizes).

In [2]:
# TODO:
# ae_train_X = ...      # only the normal class, from train_imgs (subset 1)
# val_normal_X = ...    # normal class, DIFFERENT examples than ae_train_X (subset 2)
# val_anomaly_X = ...   # other classes, from train_imgs (subset 3)
# test_normal_X = ...   # normal class, from test_imgs (subset 4)
# test_anomaly_X = ...  # other classes, from test_imgs
# test_anomaly_labels = ...  # the corresponding class labels (for the analysis in section 6)
# print the sizes, VERIFY PROGRAMMATICALLY that ae_train/val_normal/val_anomaly do not overlap
raise NotImplementedError("TODO: leak-free data split")

NotImplementedError: TODO: leak-free data split

## 2. Autoencoder -- architecture and training (provided)

A small AE (MLP), as in the lecture. We train **only** on normal data --
this is the key rule of this task, not an implementation detail.

In [3]:
class AE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(), nn.Linear(D_IN, 128), nn.ReLU(), nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(), nn.Linear(128, D_IN), nn.Sigmoid(),
        )

    def forward(self, x):
        e = self.encoder(x)
        return self.decoder(e).view(-1, 1, 28, 28), e


def train_ae(train_X, latent_dim=16, epochs=15, batch_size=64, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    model = AE(latent_dim)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = train_X.shape[0]
    t0 = time.time()
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            batch = train_X[idx]
            opt.zero_grad()
            x_hat, _ = model(batch)
            loss = F.mse_loss(x_hat, batch)
            loss.backward()
            opt.step()
    model.eval()
    print(f"training time: {time.time() - t0:.2f}s")
    return model

In [4]:
# TODO: train the AE ONLY on ae_train_X (NOT on train_imgs, NOT on a combination
# with anything else) -- use train_ae(...) from section 2 (provided), latent_dim=16.
# model = train_ae(???)
raise NotImplementedError("TODO: train the AE on normal data")

NotImplementedError: TODO: train the AE on normal data

## 3. Reconstruction error and distributions -- your task

Implement `per_image_error(model, X)` -- the reconstruction error **per
image** (not aggregated): `((x_hat - x)**2).flatten(1).mean(dim=1)`.
Compute it for `val_normal_X` and `val_anomaly_X`, print mean/std/median,
draw a histogram of both distributions on one plot.

In [5]:
# TODO:
# def per_image_error(model, X):
#     ...
# err_val_normal = per_image_error(model, val_normal_X)
# err_val_anomaly = per_image_error(model, val_anomaly_X)
# print mean/std/median for both, draw a histogram
raise NotImplementedError("TODO: reconstruction error and distributions")

NotImplementedError: TODO: reconstruction error and distributions

## 4. Choosing a threshold -- your task

Set the threshold **only** based on `err_val_normal` (e.g. mean + k*std
for a k of your choice -- justify it). **Do not** use `err_val_anomaly`.

In [6]:
# TODO: threshold = err_val_normal.mean() + k * err_val_normal.std()  (choose k, justify it)
raise NotImplementedError("TODO: choosing the threshold from val_normal")

NotImplementedError: TODO: choosing the threshold from val_normal

## 5. Metrics -- your task

**First** verify your implementation on a known example (see
`README.md`, section 5): `scores`/`labels` below, threshold $0.45$.
Expected: TP=5, FP=1, TN=4, FN=0, precision~=0.833, recall=1.0,
F1~=0.909.

In [7]:
known_scores = torch.tensor([0.1, 0.2, 0.3, 0.4, 0.9, 0.5, 0.6, 0.7, 0.8, 1.0])
known_labels = torch.tensor([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])
known_threshold = 0.45

# TODO:
# def confusion_counts(scores, labels, threshold):  # -> tp, fp, tn, fn
#     ...
# def precision_recall_f1(tp, fp, fn):
#     ...
# verify on known_scores/known_labels/known_threshold, BEFORE using it on real data
raise NotImplementedError("TODO: metrics on the known example")

NotImplementedError: TODO: metrics on the known example

In [8]:
# TODO: implement by hand (sklearn is NOT installed in this repo):
# def roc_auc(labels, scores):       # rank-based: P(score_anomaly > score_normal)
#     ...
# def pr_auc(labels, scores):        # average precision
#     ...
# verify both on known_scores/known_labels (expected: roc_auc=0.84, pr_auc=0.81)
raise NotImplementedError("TODO: ROC-AUC / PR-AUC on the known example")

NotImplementedError: TODO: ROC-AUC / PR-AUC on the known example

In [9]:
# TODO: compute err_test_normal, err_test_anomaly (per_image_error from section 3),
# then confusion_counts/precision_recall_f1/roc_auc/pr_auc on the REAL test
# data, using the threshold from section 4 (NEVER choose a threshold here!)
raise NotImplementedError("TODO: metrics on the test data")

NotImplementedError: TODO: metrics on the test data

## 6. Error analysis and a simple baseline -- your task

**A baseline with no learning**: the error = distance to the mean image
of the normal class (`ae_train_X.mean(dim=0)`), with no training at all.
Compute the same metrics for this baseline on the test data and compare
with AE -- **without** assuming in advance which one will do better.

In [10]:
# TODO:
# mean_normal_image = ae_train_X.mean(dim=0, keepdim=True)
# baseline_err_test_normal = ((test_normal_X - mean_normal_image) ** 2).flatten(1).mean(dim=1)
# baseline_err_test_anomaly = ((test_anomaly_X - mean_normal_image) ** 2).flatten(1).mean(dim=1)
# compute roc_auc for the baseline, compare with the AE roc_auc from section 5
raise NotImplementedError("TODO: trivial baseline")

NotImplementedError: TODO: trivial baseline

In [11]:
# TODO: find AT LEAST one specific case:
# 1. an anomaly with a LOW reconstruction error (false negative) -- which class? visualize it.
# 2. a normal example with a HIGH error (false positive) -- visualize it.
# (test_anomaly_labels is useful to check which class a false negative came from)
raise NotImplementedError("TODO: error analysis")

NotImplementedError: TODO: error analysis

## 7. Diagnostic task: data leakage in threshold selection

The following code (provided, run it) **looks** reasonable -- it searches
a grid of thresholds and picks the one with the best F1 -- but it
chooses **based on the test set's labels**.

In [12]:
# noqa: F821 - err_test_normal/err_test_anomaly/confusion_counts/precision_recall_f1
# defined by you in sections 3 and 5
candidate_thresholds = torch.linspace(err_test_normal.min(), err_test_anomaly.max(), 500)  # noqa: F821
best_f1_leaky, best_t_leaky = -1.0, None
for t in candidate_thresholds:
    tp_, fp_, tn_, fn_ = confusion_counts(  # noqa: F821
        torch.cat([err_test_normal, err_test_anomaly]),  # noqa: F821
        torch.cat([torch.zeros(len(err_test_normal)), torch.ones(len(err_test_anomaly))]),  # noqa: F821
        t.item(),
    )
    _, _, f1_ = precision_recall_f1(tp_, fp_, fn_)  # noqa: F821
    if f1_ > best_f1_leaky:
        best_f1_leaky, best_t_leaky = f1_, t.item()

print(f"'leaky' threshold (chosen on test labels): {best_t_leaky:.5f}")
print(f"'leaky' F1 (on the same set that chose the threshold): {best_f1_leaky:.4f}")

NameError: name 'err_test_normal' is not defined

In [13]:
# TODO:
# 1. compare best_t_leaky with your threshold from section 4 -- identical?
# 2. compare best_f1_leaky with your honest F1 from section 5 (on the test set,
#    with the threshold from section 4)
# 3. in a comment: explain why this procedure is wrong REGARDLESS of whether
#    the F1 difference turned out large or small in this specific run
raise NotImplementedError("TODO: diagnosing the data leakage")

NotImplementedError: TODO: diagnosing the data leakage

## 8. Conclusions -- your task

**1. How well (specific numbers) does your AE separate the normal class from anomalies on the test set?**

_(Your answer here.)_

**2. How does it compare with the trivial baseline? What does that say about where the discriminative ability comes from?**

_(Your answer here.)_

**3. What is your most convincing specific example of the method's limitation (section 6)?**

_(Your answer here.)_

**4. What bug did the data leakage in section 7 introduce, and what general rule do you draw from it?**

_(Your answer here.)_

**5. Would you consider this system ready for practical use? What would it still need?**

_(Your answer here.)_

## Extension (optional)

See `README.md` -- the effect of latent dimension, or VAE as an
alternative. Not counted in the 115-minute budget.